In [1]:
from pathlib import Path
import re
import pandas as pd

# Folder containing the SHAP CSV files
input_folder = Path(".")

# File-name pattern:
filename_pattern = re.compile(
    r"^shap_values_cnn2_mismatch_iMeta_TTTV_filtered_(\d+)\.csv$"
)

# Find and sort all matching files by model number
matching_files = []

for file_path in input_folder.glob("shap_values_cnn2_mismatch_iMeta_TTTV_filtered_*.csv"):
    match = filename_pattern.match(file_path.name)

    if match:
        model_number = int(match.group(1))
        matching_files.append((model_number, file_path))

matching_files.sort(key=lambda item: item[0])

print(f"Detected {len(matching_files)} matching files:")
for model_number, file_path in matching_files:
    print(f"Model {model_number}: {file_path.name}")


# Process every detected CSV file
for model_number, input_path in matching_files:

    print(f"\nProcessing model {model_number}: {input_path.name}")

    # Pandas normally renames the second duplicate column:
    df = pd.read_csv(input_path)

    # Select the second occurrence of duplicated columns
    shap_df = df.loc[:, df.columns.str.endswith(".1")].copy()

    if shap_df.shape[1] == 0:
        print(
            f"Warning: No columns ending in '.1' were detected in "
            f"{input_path.name}. Skipping this file."
        )
        continue

    # Restore the original feature names
    shap_df.columns = shap_df.columns.str.replace(
        r"\.1$",
        "",
        regex=True
    )

    # Make sure values are numeric
    # Invalid entries are converted to NaN and ignored by mean()
    shap_df = shap_df.apply(pd.to_numeric, errors="coerce")

    # Importance magnitude = mean absolute SHAP value across all samples
    importance_magnitude = shap_df.abs().mean(axis=0)

    # Create output table
    importance_table = pd.DataFrame({
        "Feature": importance_magnitude.index,
        "Importance Magnitude": importance_magnitude.values
    })

    # Sort from highest to lowest importance
    importance_table = importance_table.sort_values(
        by="Importance Magnitude",
        ascending=False,
        ignore_index=True
    )

    # Output file name
    output_path = input_folder / (
        f"importance_magnitude_cnn2_mismatch_iMeta_TTTV_filtered_{model_number}.csv"
    )

    importance_table.to_csv(output_path, index=False)

    print(f"Saved: {output_path.name}")

print("\nAll matching files have been processed.")

Detected 10 matching files:
Model 0: shap_values_cnn2_mismatch_iMeta_TTTV_filtered_0.csv
Model 5: shap_values_cnn2_mismatch_iMeta_TTTV_filtered_5.csv
Model 8: shap_values_cnn2_mismatch_iMeta_TTTV_filtered_8.csv
Model 9: shap_values_cnn2_mismatch_iMeta_TTTV_filtered_9.csv
Model 11: shap_values_cnn2_mismatch_iMeta_TTTV_filtered_11.csv
Model 47: shap_values_cnn2_mismatch_iMeta_TTTV_filtered_47.csv
Model 80: shap_values_cnn2_mismatch_iMeta_TTTV_filtered_80.csv
Model 87: shap_values_cnn2_mismatch_iMeta_TTTV_filtered_87.csv
Model 96: shap_values_cnn2_mismatch_iMeta_TTTV_filtered_96.csv
Model 99: shap_values_cnn2_mismatch_iMeta_TTTV_filtered_99.csv

Processing model 0: shap_values_cnn2_mismatch_iMeta_TTTV_filtered_0.csv
Saved: importance_magnitude_cnn2_mismatch_iMeta_TTTV_filtered_0.csv

Processing model 5: shap_values_cnn2_mismatch_iMeta_TTTV_filtered_5.csv
Saved: importance_magnitude_cnn2_mismatch_iMeta_TTTV_filtered_5.csv

Processing model 8: shap_values_cnn2_mismatch_iMeta_TTTV_filtered_8